In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import glob

BASE_PATH = '/content/drive/MyDrive'

eicu_candidates = glob.glob(f'{BASE_PATH}/**/*eicu*', recursive=True, root_dir=None)
# case-insensitive fallback in case the folder is capitalized differently
import os
all_dirs = glob.glob(f'{BASE_PATH}/**/', recursive=True)
eicu_dirs = [d for d in all_dirs if 'eicu' in d.lower()]
print(eicu_dirs)

In [ ]:
import os

for f in sorted(os.listdir('/content/drive/MyDrive/eICU-CRD/')):
    full_path = os.path.join('/content/drive/MyDrive/eICU-CRD/', f)
    size_mb = os.path.getsize(full_path) / 1e6 if os.path.isfile(full_path) else None
    print(f, f"({size_mb:.1f} MB)" if size_mb else "(directory)")

In [ ]:
EICU_PATH = '/content/drive/MyDrive/eICU-CRD/'

admissiondx = pd.read_csv(EICU_PATH + 'admissionDx.csv.gz')
treatment = pd.read_csv(EICU_PATH + 'treatment.csv.gz')

print(admissiondx.columns.tolist())
print(treatment.columns.tolist())

In [ ]:
admit_matches = admissiondx[admissiondx['admitdxtext'].str.contains('aort|aneurysm', case=False, na=False)]
print(f"admissionDx matches: {admit_matches['admitdxtext'].nunique()} unique values, {len(admit_matches)} rows")
print(admit_matches['admitdxtext'].value_counts().head(30))

print("\n--- treatment ---")
treat_matches = treatment[treatment['treatmentstring'].str.contains('aort|aneurysm', case=False, na=False)]
print(f"treatment matches: {treat_matches['treatmentstring'].nunique()} unique values, {len(treat_matches)} rows")
print(treat_matches['treatmentstring'].value_counts().head(30))

In [ ]:
# genuine aortic aneurysm/dissection diagnoses only — excludes valve, CABG, cerebral/SAH, graft-bypass, and "complications of" categories
AORTIC_DX = [
    'Aneurysm, abdominal aortic',
    'Aneurysm, abdominal aortic; with dissection',
    'Aneurysm, abdominal aortic; with rupture',
    'Aneurysm, thoracic aortic',
    'Aneurysm, thoracic aortic; with dissection',
    'Aneurysm, thoracic aortic; with rupture',
    'Aneurysm, dissecting aortic',
]

dx_cohort = admissiondx[admissiondx['admitdxtext'].isin(AORTIC_DX)][['patientunitstayid', 'admitdxtext']].drop_duplicates()
print(f"Candidate stays from admissionDx: {dx_cohort['patientunitstayid'].nunique()}")

repair_stays = treatment[treatment['treatmentstring'] == 'cardiovascular|vascular surgery|aneurysm resection / repair']['patientunitstayid'].unique()
print(f"Stays with confirmed repair treatment: {len(repair_stays)}")

cohort_eicu = dx_cohort[dx_cohort['patientunitstayid'].isin(repair_stays)].copy()
print(f"\nOverlap (confirmed aortic + confirmed repair): {cohort_eicu['patientunitstayid'].nunique()}")
print(cohort_eicu['admitdxtext'].value_counts())

In [ ]:
def classify_location_eicu(dx):
    if 'thoracic' in dx:
        return 'thoracic'
    elif 'abdominal' in dx:
        return 'abdominal'
    return None  # the unqualified "dissecting aortic" cases

cohort_eicu['location'] = cohort_eicu['admitdxtext'].apply(classify_location_eicu)
cohort_eicu_final = cohort_eicu.dropna(subset=['location']).drop_duplicates(subset='patientunitstayid')

print(f"Final eICU-CRD cohort: {len(cohort_eicu_final)}")
print(cohort_eicu_final['location'].value_counts())

In [ ]:
patient = pd.read_csv(EICU_PATH + 'patient.csv.gz', usecols=['patientunitstayid', 'age', 'gender'])

cohort_eicu_final = cohort_eicu_final.merge(patient, on='patientunitstayid', how='left')

# the top-coded "> 89" string becomes 90 — a defensible, commonly used convention
# (keeps it numeric and just above the true max of 89, rather than guessing a higher number)
cohort_eicu_final['age_numeric'] = cohort_eicu_final['age'].replace('> 89', '90')
cohort_eicu_final['age_numeric'] = pd.to_numeric(cohort_eicu_final['age_numeric'], errors='coerce')

print(cohort_eicu_final['age_numeric'].describe())
print(cohort_eicu_final['gender'].value_counts(dropna=False))

In [ ]:
diagnosis = pd.read_csv(EICU_PATH + 'diagnosis.csv.gz', usecols=['patientunitstayid', 'diagnosisstring', 'icd9code'])
diag_cohort = diagnosis[diagnosis['patientunitstayid'].isin(cohort_eicu_final['patientunitstayid'])]

print(diag_cohort['icd9code'].head(20).tolist())
print(f"\nNull icd9code rate: {diag_cohort['icd9code'].isna().mean():.1%}")

In [ ]:
diag_nonnull = diag_cohort.dropna(subset=['icd9code']).copy()

split_counts = diag_nonnull['icd9code'].str.split(',').apply(len)
print("Rows by number of comma-separated parts:")
print(split_counts.value_counts())

diag_nonnull['icd10_part'] = diag_nonnull['icd9code'].str.split(',').str[1].str.strip()
print("\nSample ICD-10 parts:", diag_nonnull['icd10_part'].head(10).tolist())

stays_with_diag = diag_nonnull['patientunitstayid'].nunique()
print(f"\nStays with at least one usable code: {stays_with_diag} / {len(cohort_eicu_final)}")

In [ ]:
print("1-part examples:")
print(diag_nonnull[split_counts == 1]['icd9code'].head(10).tolist())
print("\n3-part examples:")
print(diag_nonnull[split_counts == 3]['icd9code'].head(10).tolist())

In [ ]:
!pip install comorbidipy --quiet
import polars as pl
from comorbidipy import comorbidity

In [ ]:
def pick_code(cell):
    parts = [p.strip() for p in cell.split(',')]
    icd10_parts = [p for p in parts if p[:1].isalpha()]
    if icd10_parts:
        return icd10_parts[0], 10
    return parts[0], 9

diag_nonnull[['picked_code', 'picked_version']] = diag_nonnull['icd9code'].apply(lambda c: pd.Series(pick_code(c)))

diag9_e = pl.from_pandas(diag_nonnull[diag_nonnull['picked_version'] == 9][['patientunitstayid', 'picked_code']].rename(columns={'picked_code': 'icd_code'}))
diag10_e = pl.from_pandas(diag_nonnull[diag_nonnull['picked_version'] == 10][['patientunitstayid', 'picked_code']].rename(columns={'picked_code': 'icd_code'}))

cci9_e = comorbidity(diag9_e, id_col='patientunitstayid', code_col='icd_code', icd='icd9')
cci10_e = comorbidity(diag10_e, id_col='patientunitstayid', code_col='icd_code', icd='icd10')
cci_eicu = pl.concat([cci9_e, cci10_e], how='diagonal').to_pandas()[['patientunitstayid', 'comorbidity_score']]
cci_eicu = cci_eicu.rename(columns={'comorbidity_score': 'cci'})

cohort_eicu_final = cohort_eicu_final.merge(cci_eicu, on='patientunitstayid', how='left')
cohort_eicu_final['has_diagnosis_data'] = cohort_eicu_final['cci'].notna().astype(int)
cohort_eicu_final['cci'] = cohort_eicu_final['cci'].fillna(0)

print(f"Stays with diagnosis data: {cohort_eicu_final['has_diagnosis_data'].sum()} / {len(cohort_eicu_final)}")
print(cohort_eicu_final['cci'].describe())

In [ ]:
def extract_icd10(cell):
    parts = [p.strip() for p in cell.split(',')]
    icd10_parts = [p for p in parts if p[:1].isalpha()]
    return icd10_parts[0] if icd10_parts else None

diag_nonnull['icd10_code'] = diag_nonnull['icd9code'].apply(extract_icd10)
diag_icd10_only = diag_nonnull.dropna(subset=['icd10_code'])
print(f"Rows with a usable ICD-10 code: {len(diag_icd10_only)} / {len(diag_nonnull)} (dropped: {len(diag_nonnull) - len(diag_icd10_only)} — ICD-9-only rows with no ICD-10 equivalent, a small accepted loss)")

diag10_e = pl.from_pandas(diag_icd10_only[['patientunitstayid', 'icd10_code']].rename(columns={'icd10_code': 'icd_code'}))
cci_eicu_raw = comorbidity(diag10_e, id_col='patientunitstayid', code_col='icd_code', icd='icd10')
cci_eicu = cci_eicu_raw.to_pandas()[['patientunitstayid', 'comorbidity_score']].rename(columns={'comorbidity_score': 'cci'})

print(f"Unique stays scored: {cci_eicu['patientunitstayid'].nunique()}, total rows: {len(cci_eicu)}")  # must match exactly — confirms no duplication

In [ ]:
cohort_eicu_final = cohort_eicu_final.drop(columns=['cci', 'has_diagnosis_data'], errors='ignore')
cohort_eicu_final = cohort_eicu_final.merge(cci_eicu, on='patientunitstayid', how='left')
cohort_eicu_final['has_diagnosis_data'] = cohort_eicu_final['cci'].notna().astype(int)
cohort_eicu_final['cci'] = cohort_eicu_final['cci'].fillna(0)

print(f"Final cohort size (must be 395): {len(cohort_eicu_final)}")
print(f"Stays with diagnosis data: {cohort_eicu_final['has_diagnosis_data'].sum()} / {len(cohort_eicu_final)}")
print(cohort_eicu_final['cci'].describe())

In [ ]:
cohort_eicu_final = cohort_eicu_final.drop(columns=['cci', 'has_diagnosis_data'], errors='ignore')
cohort_eicu_final = cohort_eicu_final.drop_duplicates(subset='patientunitstayid')
print(f"After dedup (should be 395): {len(cohort_eicu_final)}")

cohort_eicu_final = cohort_eicu_final.merge(cci_eicu, on='patientunitstayid', how='left')
cohort_eicu_final['has_diagnosis_data'] = cohort_eicu_final['cci'].notna().astype(int)
cohort_eicu_final['cci'] = cohort_eicu_final['cci'].fillna(0)

print(f"Final cohort size (must be 395): {len(cohort_eicu_final)}")
print(f"Stays with diagnosis data: {cohort_eicu_final['has_diagnosis_data'].sum()} / {len(cohort_eicu_final)}")
print(cohort_eicu_final['cci'].describe())

In [ ]:
has_data = cohort_eicu_final[cohort_eicu_final['has_diagnosis_data'] == 1]
print(f"Mean CCI among the 278 with diagnosis data: {has_data['cci'].mean():.3f}")
print(has_data['cci'].value_counts().sort_index())

In [ ]:
vaso_treat = treatment[treatment['treatmentstring'].str.contains('norepinephrine|epinephrine|vasopressin|phenylephrine|dopamine|dobutamine', case=False, na=False, regex=True)]
print(f"Unique treatment strings: {vaso_treat['treatmentstring'].nunique()}")
print(vaso_treat['treatmentstring'].value_counts().head(20))

In [ ]:
vaso_mask = treatment['treatmentstring'].str.contains('norepinephrine|epinephrine|vasopressin|phenylephrine|dopamine|dobutamine', case=False, na=False, regex=True)
gi_exclude_mask = treatment['treatmentstring'].str.startswith('gastrointestinal')

vaso_final = treatment[vaso_mask & ~gi_exclude_mask]
vaso_stays_eicu = vaso_final['patientunitstayid'].unique()

cohort_eicu_final['vasopressor'] = cohort_eicu_final['patientunitstayid'].isin(vaso_stays_eicu).astype(int)
print(cohort_eicu_final['vasopressor'].value_counts())
print(f"Vasopressor use rate: {cohort_eicu_final['vasopressor'].mean():.1%}")

In [ ]:
treat_coverage = treatment[treatment['patientunitstayid'].isin(cohort_eicu_final['patientunitstayid'])]
stays_with_any_treatment = treat_coverage['patientunitstayid'].nunique()
print(f"Stays with ANY treatment table entry: {stays_with_any_treatment} / {len(cohort_eicu_final)}")

# among stays that DO have treatment data, what's the vasopressor rate specifically?
has_treatment_ids = treat_coverage['patientunitstayid'].unique()
subset = cohort_eicu_final[cohort_eicu_final['patientunitstayid'].isin(has_treatment_ids)]
print(f"Vasopressor rate among stays WITH treatment data: {subset['vasopressor'].mean():.1%}")

In [ ]:
labname_sample = set()
for chunk in pd.read_csv(EICU_PATH + 'lab.csv.gz', usecols=['labname'], chunksize=2_000_000):
    labname_sample.update(chunk['labname'].dropna().unique())

creat_names = [n for n in labname_sample if 'creat' in n.lower()]
print(creat_names)

In [ ]:
CHUNK_SIZE = 2_000_000
cohort_stays = set(cohort_eicu_final['patientunitstayid'])
chunks = []

for chunk in pd.read_csv(EICU_PATH + 'lab.csv.gz',
                          usecols=['patientunitstayid', 'labname', 'labresultoffset', 'labresult'],
                          chunksize=CHUNK_SIZE):
    filtered = chunk[(chunk['labname'] == 'creatinine') & (chunk['patientunitstayid'].isin(cohort_stays))]
    if len(filtered):
        chunks.append(filtered)

creat_eicu = pd.concat(chunks, ignore_index=True)
print(f"Creatinine values pulled: {len(creat_eicu)}")
print(f"Stays with at least one value: {creat_eicu['patientunitstayid'].nunique()} / {len(cohort_eicu_final)}")

In [ ]:
baseline_window_eicu = creat_eicu[creat_eicu['labresultoffset'] <= 1440]  # first 24h = 1440 minutes from ICU admission
baseline_eicu = baseline_window_eicu.groupby('patientunitstayid')['labresult'].min().rename('baseline_creat')

peak_eicu = creat_eicu.groupby('patientunitstayid')['labresult'].max().rename('peak_creat')

aki_eicu = pd.concat([baseline_eicu, peak_eicu], axis=1).reset_index()
aki_eicu['aki'] = (aki_eicu['peak_creat'] >= 1.5 * aki_eicu['baseline_creat']).astype(int)

cohort_eicu_final = cohort_eicu_final.merge(aki_eicu[['patientunitstayid', 'aki']], on='patientunitstayid', how='left')

print(f"Stays with usable AKI calculation: {cohort_eicu_final['aki'].notna().sum()} / {len(cohort_eicu_final)}")
print(cohort_eicu_final['aki'].value_counts(dropna=False))
print(f"AKI rate (among those with data): {cohort_eicu_final['aki'].mean():.1%}")

cohort_eicu_final.to_csv('/content/drive/MyDrive/eICU-CRD/aortic_aneurysm_eicu_cohort_final.csv', index=False)

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

cohort_model_eicu = cohort_eicu_final.dropna(subset=['aki']).copy()
print(f"Modeling dataset: {len(cohort_model_eicu)} (excluded {len(cohort_eicu_final) - len(cohort_model_eicu)} with no creatinine data)")

model_eicu = smf.logit(
    'aki ~ C(location, Treatment(reference="abdominal")) '
    '+ age_numeric + C(gender) + cci + vasopressor',
    data=cohort_model_eicu
).fit()

print(model_eicu.summary())

odds_ratios_eicu = pd.DataFrame({
    'OR': np.exp(model_eicu.params),
    'CI_lower': np.exp(model_eicu.conf_int()[0]),
    'CI_upper': np.exp(model_eicu.conf_int()[1]),
    'p_value': model_eicu.pvalues
})
print(odds_ratios_eicu)

In [ ]:
clean_dx = ['Aneurysm, abdominal aortic', 'Aneurysm, thoracic aortic']

cohort_clean = cohort_model_eicu[cohort_model_eicu['admitdxtext'].isin(clean_dx)].copy()
print(f"Clean subset: {len(cohort_clean)} (dropped {len(cohort_model_eicu) - len(cohort_clean)} dissection/rupture cases)")
print(cohort_clean['location'].value_counts())
print(cohort_clean.groupby('location')['aki'].agg(['mean', 'count']))

In [ ]:
hospital = pd.read_csv(EICU_PATH + 'hospital.csv.gz')
print(hospital.columns.tolist())
print(hospital[['numbedscategory', 'teachingstatus']].value_counts() if 'numbedscategory' in hospital.columns else hospital.head())

In [ ]:
patient_hosp = pd.read_csv(EICU_PATH + 'patient.csv.gz', usecols=['patientunitstayid', 'hospitalid'])
cohort_hosp = cohort_clean.merge(patient_hosp, on='patientunitstayid', how='left')
cohort_hosp = cohort_hosp.merge(hospital[['hospitalid', 'numbedscategory', 'teachingstatus']], on='hospitalid', how='left')

print(pd.crosstab(cohort_hosp['location'], cohort_hosp['numbedscategory']))
print()
print(pd.crosstab(cohort_hosp['location'], cohort_hosp['teachingstatus']))